# IBGE Municipalities - Bronze Ingestion

## Parameters

In [0]:
dbutils.widgets.text(name="environment", defaultValue="dev", label="Environment")

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
import requests
from datetime import datetime
from pyspark.sql.functions import col, current_timestamp, lit
import uuid

In [0]:
catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "ibge_municipalities"
target_table = f"{catalog}.{schema}.{table_name}"
source_system = "ibge"
source_dataset = "municipalities"

run_id = str(uuid.uuid4())

api_url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

## Call the IBGE API

In [0]:
response = requests.get(api_url, timeout=5)

if response.status_code != 200:
    raise RuntimeError(f"Expected 200, got {response.status_code}")

## Save response to volume

In [0]:
raw_directory = (
    f"/Volumes/{catalog}/landing/raw_files/api/{source_dataset}"
)

dbutils.fs.mkdirs(raw_directory)

volume_path = (
    f"{raw_directory}/"
    f"municipalities_api_response_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

with open(volume_path, "w", encoding="utf-8") as f:
    f.write(response.text)

In [0]:
source_df = spark.read.json(volume_path)

In [0]:
bronze_df = source_df.select(
    col("id").cast("long").alias("ibge_municipality_id"),
    col("nome").alias("municipality_name"),
    col("regiao-imediata.regiao-intermediaria.UF.sigla").alias("state_code"),
    col("_metadata.file_path").alias("source_file_path"),
    col("_metadata.file_modification_time").alias(
        "source_file_modification_time"
    )
)

## Add Bronze metadata

In [0]:
bronze_df = (
    bronze_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to Bronze

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)